Orbits Homework 4

In [2]:
# Imports
import Orbits_Functions as of
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt

Problem 8.2

In [3]:
rj = 778.5e6
rm = 228e6

dv1, dv2, dv_tot, ToF = of.hohmann_transfer(rm, rj, 1.32712e11)
print(dv_tot)

10.149285317140986


Problem 8.4

In [4]:
Tj = 4333 # days
Tm = 687 # days

Tsyn = of.synodic_period(Tj, Tm)
print(Tsyn/365)

2.236845031898346


Problem 8.6

In [5]:
# Constants
a = [1.4291e9, 2.87479e9, 4.50489e9, 5.9154e9]
mu = [3.79406e7, 5.79456e6, 6.83653e6, 9.755e2]
labels = ['Saturn', 'Uranus', 'Neptune', 'Pluto']

# Analysis
soi = []  # store results

for i in range(len(a)):
    soi.append(of.sphere_of_influence(a[i], mu[i]))

# Print
for i in range(len(a)):
    print(f"The sphere of influence of {labels[i]} is {soi[i]} km")

The sphere of influence of Saturn is 54643715.23358699 km
The sphere of influence of Uranus is 51838238.26239671 km
The sphere of influence of Neptune is 86786945.36274141 km
The sphere of influence of Pluto is 3299977.272540446 km


Problem 8.7

In [6]:
ra = 147.4e6
rp = 120e6
mu_s = 1.32712e11

a = (ra+rp)/2
va = np.sqrt(mu_s*(2/ra - 1/a))

r_e = 147.4e6
ve = np.sqrt(mu_s*(2/r_e - 1/r_e))

vinf = ve - va

v_park = np.sqrt(398600/6578)
v_hyp = np.sqrt(vinf**2 + (2*398600)/6578)

dv = v_hyp - v_park

print(dv)
print(vinf)

3.337022042198777
1.5788630668882107


Problem 8.12

In [7]:
# Constants
mu_e = 398600
mu_s = 1.327e11
mu_j = 1.267e8

rearth = 6378
rs2j = 778.6e6
rs2e = 149.6e6
rjupiter = 71490

# Hohmann transfer
rp_t = rs2e
ra_t = rs2j
a_t = (rp_t+ra_t)/2

vp_t = np.sqrt(mu_s * (2/rp_t - 1/a_t))
va_t = np.sqrt(mu_s * (2/ra_t - 1/a_t))

v_e = np.sqrt(mu_s/rs2e)
v_j = np.sqrt(mu_s/rs2j)

v_inf_e = vp_t - v_e
v_inf_j = v_j - va_t
v_inf = abs(v_inf_j)

# Hyperbolic flyby around Jupiter
rp_j = rjupiter + 200000

e_hyp = 1 + (rp_j * v_inf**2)/mu_j
turn_ang = 2 * np.arcsin(1/e_hyp)

# Arrival v_inf in Jupiter frame
v_inf_arr = np.array([0.0, -v_inf])
theta_in = -np.pi/2

# Angle of outgoing
theta_out = theta_in - turn_ang
v_inf_out = v_inf*np.array([np.cos(theta_out), np.sin(theta_out)])

# Adding jupiter velocity
v_J_vec = np.array(0.0, v_j)

v_sc_in = v_J_vec + v_inf_arr
v_sc_out = v_J_vec + v_inf_out

dv = np.linalg.norm(v_sc_out - v_sc_in)

print(dv)

10.565094561268895


Problem 8.16

In [ ]:
# Constants
mu_s = 1.327e11
mu_m = 42828
T_m = 35*3600

# Time calculations
t_i = of.julian_date(2005, 8, 15, 0, 0, 0, False)
t_f = of.julian_date(2006, 3, 15, 0, 0, 0, False)
jcent_i = (t_i-2451545)/36525   # Julian century
jcent_f = (t_f-2451545)/36525   # Julian century

dt = t_f - t_i  # days

# Heliocentric positions and velocities
COE_e_i = of.planetary_elements(3, jcent_i)
COE_m_f = of.planetary_elements(4, jcent_f)

w_hat_e = np.deg2rad(COE_e_i['w_hat'])
raan_e = np.deg2rad(COE_e_i['raan'])
L_e = np.deg2rad(COE_e_i['L'])
ecc_e = COE_e_i['ecc']
a_e = COE_e_i['a']
inc_e = np.deg2rad(COE_e_i['inc'])

w_hat_m = np.deg2rad(COE_m_f['w_hat'])
raan_m = np.deg2rad(COE_m_f['raan'])
L_m = np.deg2rad(COE_m_f['L'])
ecc_m = COE_m_f['ecc']
a_m = COE_m_f['a']
inc_m = np.deg2rad(COE_m_f['inc'])

argp_e_i = w_hat_e - raan_e
argp_m_f = w_hat_m - raan_m

meana_e_i = L_e - w_hat_e
meana_m_f = L_m - w_hat_m

TA_e_i = of.mean_to_true_anomaly(meana_e_i, ecc_e)
TA_m_f = of.mean_to_true_anomaly(meana_m_f, ecc_m)

p_e_i = a_e*(1-ecc_e**2)
p_m_f = a_m*(1-ecc_m**2)

r_e_i = p_e_i/(1+ecc_e*np.cos(TA_e_i))
r_m_f = p_m_f/(1+ecc_m*np.cos(TA_m_f))

v_e_i = np.sqrt(mu_s*(2/r_e_i - 1/a_e))
v_m_f = np.sqrt(mu_s*(2/r_m_f - 1/a_m))

h_e_i = r_e_i * v_e_i
h_m_f = r_m_f * v_m_f

rvec_e, vvec_e = of.COEs2ECI(h_e_i, ecc_e, inc_e, raan_e, argp_e_i, TA_e_i, mu=mu_s)
rvec_m, vvec_m = of.COEs2ECI(h_m_f, ecc_m, inc_m, raan_m, argp_m_f, TA_m_f, mu=mu_s)

# Lamberts solution and hyperbolic excess velocity
vvec_i, vvec_f, orb = of.lamberts(rvec_e, rvec_m, dt*86400, prograde=True, max_iter=25, tol=1e-8, mu=mu_s)

vinf_e = vvec_i - vvec_e
vinf_m = vvec_f - vvec_m


# Delta v to depart from earth
r_park = 6378 + 190
v_park = np.sqrt(mu_e/r_park)
vp_hyp_e = np.sqrt(np.linalg.norm(vinf_e)**2+(2*mu_e)/r_park)

dv1 = vp_hyp_e - v_park


# Delta v to 300km periapsis in mars
rp_m = 3390 + 300
a_m = (mu_m**(1/3))*(T_m/(2*np.pi))**(2/3)
vp_m = np.sqrt(mu_m*(2/rp_m - 1/a_m))


# Delta v to capture at mars
vp_hyp_m = np.sqrt(np.linalg.norm(vinf_m)**2 + (2*mu_m)/rp_m)

dv2 = vp_hyp_m - vp_m

# Total delta v
dvtot = dv1 + dv2
print(dvtot)


[ 1.19745032e+08 -9.28634931e+07 -1.18894424e+03] [1.77684545e+01 2.34257131e+01 2.99922668e-04] [-8.37055917e+07  2.27919491e+08  6.83141872e+06] [-21.76824512  -6.27844942   0.40320013]
4.907058512745084
